# Recommendation Engine (merged, final)

This merges your colleague's latest `recommendation_engine.ipynb` with the
version built earlier in this project. What changed and why:

* **Skill canonicalization** (`skill_aliases` / `canonicalize_skill`) is new
  here — "ML", "Machine Learning", and "Machine Learning Algorithms" now all
  collapse to one skill before anything is scored. Adopted as-is from your
  colleague's notebook.
* **Weighted skill-gap coverage is now the same idea as before, just
  cleaner**: your colleague's `calculate_weighted_skill_gap_coverage` and the
  version written earlier do the same thing (each covered skill contributes
  its weight once). Kept the earlier one's behaviour of defaulting a missing
  skill to weight 1.0, since that's what lets `get_recommendations` be called
  with no weights at all and still work.
* **Semantic similarity now uses `BAAI/bge-base-en-v1.5`** instead of
  `all-MiniLM-L6-v2`, and builds a richer semantic query out of the stated
  `career_goal` and a free-text `learner_description`, not just the bare
  skill-gap keywords. This is a real quality improvement, adopted as-is.
* **Hybrid weights updated to 0.40 coverage / 0.45 semantic / 0.15 TF-IDF**,
  matching your colleague's latest notebook (previously 0.50/0.30/0.20).
* **Rank + per-signal contribution breakdown** in the final output is new —
  adopted from your colleague's notebook, and useful for the agent's
  "explain" step.
* **Still added on top, because neither notebook had them yet:**
  metadata-aware adjustment (course ratings, missing treated as unknown not
  zero) and diversity filtering (drops near-duplicate courses by skill
  overlap) — Step 4C from the original project plan.
* **The "Semantic Embedding Generator Model ka Analysis (Not For Use)"
  section at the end of your colleague's notebook was left out** — it's
  explicitly marked as scratch analysis, not part of the pipeline.
* **Still wired to a real `person_id`** the same way as before, so the
  agent doesn't have to.


## Step 1 — Setup and data loading

In [1]:
import ast

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.width", 160)

user_profiles = pd.read_csv("data/recommendation_ready/user_profiles.csv")
course_profiles = pd.read_csv("data/recommendation_ready/course_profiles.csv")

print("User Profiles Shape  :", user_profiles.shape)
print("Course Profiles Shape:", course_profiles.shape)

User Profiles Shape  : (54933, 13)
Course Profiles Shape: (404, 23)


In [2]:
def parse_list_cell(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return parsed
    except (ValueError, SyntaxError):
        pass
    return [s.strip() for s in str(value).split(",") if s.strip()]

course_profiles["skills_list"] = course_profiles["skills_list"].apply(parse_list_cell)
user_profiles["skills"] = user_profiles["skills"].apply(parse_list_cell)

print(course_profiles.loc[0, "skills_list"][:5])
print(user_profiles.loc[0, "skills"][:5])

['network security', 'python programming', 'linux', 'cloud computing', 'algorithms']
['database administration', 'database', 'ms sql server', 'ms sql server 2005', 'sql server']


## Step 2 — Skill normalization and canonicalization

Two layers, both kept: `normalize_skill` (from `02_recommendation_data.ipynb`,
lowercases and strips punctuation) and `canonicalize_skill` on top of it
(your colleague's addition, collapsing known equivalent names). Extend
`skill_aliases` as you find more equivalent skill names in the data.

In [3]:
def normalize_skill(skill):
    if pd.isna(skill):
        return None
    skill = str(skill).lower().strip()
    skill = skill.replace("&", " and ")
    skill = skill.replace("-", " ")
    skill = " ".join(skill.split())
    return skill if skill else None


# Known skill aliases / equivalent names - extend this as you find more
skill_aliases = {
    "python": "python",
    "python programming": "python",

    "ml": "machine learning",
    "machine learning": "machine learning",
    "machine learning algorithms": "machine learning",

    "sql": "sql",

    "statistics": "statistics",
    "general statistics": "statistics",
    "probability and statistics": "statistics",

    "tableau": "tableau",
    "tableau software": "tableau",
}


def canonicalize_skill(skill):
    normalized = normalize_skill(skill)
    if normalized is None:
        return None
    return skill_aliases.get(normalized, normalized)


course_profiles["skills_canonical"] = course_profiles["skills_list"].apply(
    lambda skills: [canonicalize_skill(s) for s in skills]
)

print(course_profiles.loc[0, ["skills_list", "skills_canonical"]])

skills_list         [network security, python programming, linux, ...
skills_canonical    [network security, python, linux, cloud comput...
Name: 0, dtype: object


## Step 3 — Pull a real learner's current skills

In [4]:
def get_user_current_skills(person_id, user_profiles=user_profiles):
    row = user_profiles.loc[user_profiles["person_id"] == person_id]
    if row.empty:
        raise ValueError(f"No user found with person_id={person_id}")
    row = row.iloc[0]
    current_skills = [canonicalize_skill(s) for s in row["skills"]]
    current_skills = [s for s in current_skills if s]
    return {
        "person_id": row["person_id"],
        "name": row["name"],
        "career_context": row["career_context"],
        "current_skills": current_skills,
    }


test_learner = get_user_current_skills(1)
print("Name           :", test_learner["name"])
print("Career context :", test_learner["career_context"])
print("Current skills :", test_learner["current_skills"])

Name           : Database Administrator
Career context : database administrator
Current skills : ['database administration', 'database', 'ms sql server', 'ms sql server 2005', 'sql server', 'sql server 2005', 'sql server 2008', 'sql server 2008 r2', 'sql server 2012', 'sql', 'sql queries', 'stored procedures', 'clustering', 'backups', 't sql', 'virtualization', 'r2', 'maintenance', 'problem solving', 'shipping']


## Step 4 — Skill gap and weighted skill-gap coverage

In [5]:
def compute_skill_gap(current_skills, target_skills):
    current_canonical = {canonicalize_skill(s) for s in current_skills}
    target_canonical = [canonicalize_skill(s) for s in target_skills]
    target_canonical = [s for s in target_canonical if s]
    return [s for s in target_canonical if s not in current_canonical]


def compute_weighted_skill_gap_coverage(course_profiles, skill_gap, skill_weights=None):
    """
    skill_weights: optional {skill: weight}. A gap skill missing from the
    dict defaults to weight 1.0 - so no weights at all behaves exactly like
    plain unweighted coverage.
    """
    if skill_weights is None:
        skill_weights = {}

    weights = {s: skill_weights.get(s, 1.0) for s in skill_gap}
    total_weight = sum(weights.values())

    if not skill_gap or total_weight == 0:
        return pd.Series(0.0, index=course_profiles.index)

    def coverage_for_course(course_skills):
        course_skill_set = set(course_skills)
        covered_weight = sum(w for s, w in weights.items() if s in course_skill_set)
        return covered_weight / total_weight

    return course_profiles["skills_canonical"].apply(coverage_for_course)

In [6]:
skill_gap_example = compute_skill_gap(
    test_learner["current_skills"],
    ["python", "sql", "data science", "machine learning", "cloud computing"],
)
print("Skill gap:", skill_gap_example)

unweighted = compute_weighted_skill_gap_coverage(course_profiles, skill_gap_example)
weighted = compute_weighted_skill_gap_coverage(
    course_profiles, skill_gap_example,
    skill_weights={"machine learning": 3.0, "python": 2.0, "cloud computing": 0.5},
)

comparison = course_profiles[["course_id", "title"]].copy()
comparison["coverage_unweighted"] = unweighted
comparison["coverage_weighted"] = weighted
comparison = comparison[comparison["coverage_unweighted"] > 0]
print(comparison.sort_values("coverage_weighted", ascending=False).head(10).to_string(index=False))

Skill gap: ['python', 'data science', 'machine learning', 'cloud computing']
  course_id                            title  coverage_unweighted  coverage_weighted
COURSE_0016    IBM & Darden Digital Strategy                 1.00           1.000000
COURSE_0003                 IBM Data Science                 1.00           1.000000
COURSE_0005                 IBM Data Analyst                 1.00           1.000000
COURSE_0010                   IBM Applied AI                 1.00           1.000000
COURSE_0012     Introduction to Data Science                 1.00           1.000000
COURSE_0014             IBM Data Engineering                 1.00           1.000000
COURSE_0019             Applied Data Science                 0.75           0.923077
COURSE_0036               IBM AI Engineering                 0.75           0.923077
COURSE_0287                  GPU Programming                 0.75           0.923077
COURSE_0185 Machine Learning on Google Cloud                 0.75        

## Step 5 — TF-IDF similarity

Each canonical skill is treated as one whole token (joined with a
separator), so "machine learning" doesn't get split into two words that
could spuriously match unrelated courses.

In [7]:
SKILL_SEPARATOR = " ||| "


def compute_tfidf_similarity(course_profiles, skill_gap):
    if not skill_gap:
        return pd.Series(0.0, index=course_profiles.index)

    def skill_tokenizer(text):
        return text.split(SKILL_SEPARATOR)

    course_profiles["tfidf_skill_text"] = course_profiles["skills_canonical"].apply(
        lambda skills: SKILL_SEPARATOR.join(skills)
    )

    vectorizer = TfidfVectorizer(tokenizer=skill_tokenizer, preprocessor=None,
                                  token_pattern=None, lowercase=False)
    course_matrix = vectorizer.fit_transform(course_profiles["tfidf_skill_text"])
    gap_text = SKILL_SEPARATOR.join(skill_gap)
    gap_vector = vectorizer.transform([gap_text])

    sims = cosine_similarity(gap_vector, course_matrix).flatten()
    return pd.Series(sims, index=course_profiles.index)


tfidf_scores = compute_tfidf_similarity(course_profiles, skill_gap_example)
print(tfidf_scores.describe())

count    404.000000
mean       0.081378
std        0.117793
min        0.000000
25%        0.000000
50%        0.000000
75%        0.136173
max        0.607216
dtype: float64


## Step 6 — Semantic similarity (BAAI/bge-base-en-v1.5, optional)

Builds a richer semantic query than raw skill keywords: `career_goal` +
skill gap + a free-text `learner_description` of what they're trying to
achieve. Same graceful fallback as before — if the model isn't available,
this returns `None` and the hybrid score automatically redistributes its
weight across the other two signals.

In [8]:
try:
    from sentence_transformers import SentenceTransformer
    EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"
    semantic_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    print("Semantic model loaded - semantic similarity is ENABLED.")
except Exception as e:
    semantic_model = None
    print("Semantic model not available in this environment"
          " - semantic similarity will be SKIPPED for this run.")
    print("Reason:", repr(e))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Semantic model loaded - semantic similarity is ENABLED.


In [9]:
def build_course_semantic_text(row):
    return (
        f"Title: {row['title']}. "
        f"Skills: {', '.join(row['skills_canonical'])}. "
        f"Description: {row.get('course_description_clean', '')}"
    )


def compute_semantic_similarity(course_profiles, skill_gap, model,
                                 career_goal="", learner_description=""):
    if model is None or not skill_gap:
        return None

    if "semantic_text" not in course_profiles.columns:
        course_profiles["semantic_text"] = course_profiles.apply(build_course_semantic_text, axis=1)

    learner_semantic_text = (
        f"Career Goal: {career_goal}. "
        f"Required Skills: {', '.join(skill_gap)}. "
        f"Description: {learner_description}"
    )

    course_embeddings = model.encode(course_profiles["semantic_text"].tolist(), normalize_embeddings=True)
    learner_embedding = model.encode([learner_semantic_text], normalize_embeddings=True)

    sims = cosine_similarity(learner_embedding, course_embeddings).flatten()
    return pd.Series(sims, index=course_profiles.index)

## Step 7 — Candidate filtering, normalization, and hybrid score

Weights updated to match your colleague's latest notebook: 40% weighted
skill-gap coverage, 45% semantic similarity, 15% TF-IDF.

In [10]:
def filter_candidates(scores_df, semantic_available):
    if semantic_available:
        mask = ((scores_df["skill_gap_coverage"] > 0) | (scores_df["tfidf_similarity"] > 0)
                | (scores_df["semantic_similarity"] >= 0.40))
    else:
        mask = (scores_df["skill_gap_coverage"] > 0) | (scores_df["tfidf_similarity"] > 0)
    return scores_df[mask].copy()


def normalize_scores(candidates, columns):
    scaler = MinMaxScaler()
    normed = scaler.fit_transform(candidates[columns])
    for i, col in enumerate(columns):
        candidates[f"{col}_norm"] = normed[:, i]
    return candidates


def compute_hybrid_score(candidates, semantic_available,
                          coverage_weight=0.40, semantic_weight=0.45, tfidf_weight=0.15):
    if semantic_available:
        candidates = normalize_scores(candidates, ["skill_gap_coverage", "tfidf_similarity", "semantic_similarity"])
        candidates["hybrid_score"] = (
            coverage_weight * candidates["skill_gap_coverage_norm"]
            + semantic_weight * candidates["semantic_similarity_norm"]
            + tfidf_weight * candidates["tfidf_similarity_norm"]
        )
    else:
        # semantic model unavailable - redistribute its weight proportionally
        candidates = normalize_scores(candidates, ["skill_gap_coverage", "tfidf_similarity"])
        remaining = coverage_weight + tfidf_weight
        candidates["hybrid_score"] = (
            (coverage_weight / remaining) * candidates["skill_gap_coverage_norm"]
            + (tfidf_weight / remaining) * candidates["tfidf_similarity_norm"]
        )
    return candidates

## Step 8 — Metadata adjustment and diversity filtering

Not present in either source notebook yet - this is Step 4C from the
original project plan. Ratings nudge the score by a small amount, with
missing ratings imputed to the median rather than punished as zero.
Diversity filtering then drops courses whose skill set overlaps too much
with one already selected.

In [11]:
def apply_metadata_adjustment(candidates, course_profiles, metadata_weight=0.10):
    ratings = course_profiles.loc[candidates.index, "ratings"]
    ratings_filled = ratings.fillna(ratings.median())
    candidates["rating_norm"] = MinMaxScaler().fit_transform(ratings_filled.to_frame())[:, 0]
    candidates["final_score"] = ((1 - metadata_weight) * candidates["hybrid_score"]
                                  + metadata_weight * candidates["rating_norm"])
    return candidates


def diversity_filter(ranked_candidates, course_profiles, top_k=10, max_skill_overlap=0.6):
    selected, selected_skill_sets = [], []
    for idx, row in ranked_candidates.iterrows():
        skills = set(course_profiles.loc[idx, "skills_canonical"])
        too_similar = False
        for existing in selected_skill_sets:
            if not skills or not existing:
                continue
            if len(skills & existing) / len(skills | existing) >= max_skill_overlap:
                too_similar = True
                break
        if not too_similar:
            selected.append(idx)
            selected_skill_sets.append(skills)
        if len(selected) >= top_k:
            break
    return ranked_candidates.loc[selected]

## Step 9 — Put it all together

`career_goal` and `learner_description` are optional but recommended -
they make the semantic signal meaningfully better. Without them, semantic
matching falls back to the bare skill-gap keywords.

Output includes `rank` and each signal's contribution to the final score,
for the agent's "explain" step.

In [12]:
def generate_recommendations(current_skills, target_skills, skill_weights=None,
                              career_goal="", learner_description="", top_k=8,
                              semantic_model=None, metadata_weight=0.10,
                              coverage_weight=0.40, semantic_weight=0.45, tfidf_weight=0.15):
    skill_gap = compute_skill_gap(current_skills, target_skills)

    scores = pd.DataFrame(index=course_profiles.index)
    scores["skill_gap_coverage"] = compute_weighted_skill_gap_coverage(course_profiles, skill_gap, skill_weights)
    scores["tfidf_similarity"] = compute_tfidf_similarity(course_profiles, skill_gap)

    semantic_scores = compute_semantic_similarity(course_profiles, skill_gap, semantic_model,
                                                   career_goal, learner_description)
    semantic_available = semantic_scores is not None
    if semantic_available:
        scores["semantic_similarity"] = semantic_scores

    candidates = filter_candidates(scores, semantic_available)
    candidates = compute_hybrid_score(candidates, semantic_available,
                                       coverage_weight, semantic_weight, tfidf_weight)
    candidates = apply_metadata_adjustment(candidates, course_profiles, metadata_weight)

    ranked = candidates.sort_values("final_score", ascending=False)
    final = diversity_filter(ranked, course_profiles, top_k=top_k)

    result = course_profiles.loc[
        final.index, ["course_id", "title", "organization", "difficulty", "ratings", "course_url"]
    ].copy()
    result["skill_gap_coverage"] = final["skill_gap_coverage"]
    result["tfidf_similarity"] = final["tfidf_similarity"]
    if semantic_available:
        result["semantic_similarity"] = final["semantic_similarity"]
    result["final_score"] = final["final_score"]
    result = result.sort_values("final_score", ascending=False).reset_index(drop=True)
    result.insert(0, "rank", result.index + 1)

    return skill_gap, result

In [13]:
person_id = 1
target_skills = ["python", "sql", "data science", "machine learning", "cloud computing"]
career_goal = "Machine Learning Engineer"
learner_description = (
    "Move from a database administration background into machine learning "
    "engineering within about six months, building on existing SQL and "
    "database skills."
)

learner = get_user_current_skills(person_id)
skill_gap, recommendations = generate_recommendations(
    learner["current_skills"], target_skills,
    career_goal=career_goal, learner_description=learner_description,
    top_k=8, semantic_model=semantic_model,
)

print("Learner        :", learner["name"])
print("Career context :", learner["career_context"])
print("Skill gap      :", skill_gap)
print()
print(recommendations.to_string(index=False))

Learner        : Database Administrator
Career context : database administrator
Skill gap      : ['python', 'data science', 'machine learning', 'cloud computing']

 rank   course_id                                                               title                                    organization   difficulty  ratings                                                                                                                       course_url  skill_gap_coverage  tfidf_similarity  semantic_similarity  final_score
    1 COURSE_0012                                        Introduction to Data Science                                             IBM     Beginner      4.6                                                               https://www.coursera.org/specializations/introduction-data-science                1.00          0.360548             0.793930     0.893482
    2 COURSE_0014                                                IBM Data Engineering                                     

In [14]:
# Same learner, with explicit skill weights this time
skill_weights = {
    "python": 2.0, "data science": 2.0, "machine learning": 3.0, "cloud computing": 0.5,
}

skill_gap, recommendations_weighted = generate_recommendations(
    learner["current_skills"], target_skills, skill_weights=skill_weights,
    career_goal=career_goal, learner_description=learner_description,
    top_k=8, semantic_model=semantic_model,
)
print(recommendations_weighted.to_string(index=False))

 rank   course_id                                                               title                                    organization   difficulty  ratings                                                                                                                       course_url  skill_gap_coverage  tfidf_similarity  semantic_similarity  final_score
    1 COURSE_0012                                        Introduction to Data Science                                             IBM     Beginner      4.6                                                               https://www.coursera.org/specializations/introduction-data-science            1.000000          0.360548             0.793930     0.893482
    2 COURSE_0014                                                IBM Data Engineering                                             IBM     Beginner      4.6                                                             https://www.coursera.org/professional-certificates/ibm-data-engineer  

## What's still open

* `target_skills`, `skill_weights`, `career_goal`, and `learner_description`
  are all still passed in manually here. The agent notebook is what's meant
  to supply these from a real conversation.
* `skill_aliases` only covers a handful of skills so far - worth extending
  as you notice more equivalent names in the data.
* `max_skill_overlap` (0.6) and `metadata_weight` (0.10) are starting
  values, not validated ones.
